# Data Quality Check / Data Profiling

In [2]:
from xml.etree.ElementInclude import include

import duckdb
import pandas as pd
import numpy as np

In [3]:
DATABASE_PATH = "../healthcare.duckdb"

connection = duckdb.connect(DATABASE_PATH)

In [4]:
df_raw = connection.execute("SELECT * FROM raw_analytical_dataset").fetch_df()


In [5]:
df_raw.shape

(500000, 51)

In [6]:
df_raw.info()

<class 'pandas.DataFrame'>
RangeIndex: 500000 entries, 0 to 499999
Data columns (total 51 columns):
 #   Column                Non-Null Count   Dtype  
---  ------                --------------   -----  
 0   patient_id            500000 non-null  str    
 1   patient_name          500000 non-null  str    
 2   age                   500000 non-null  int64  
 3   gender                475124 non-null  str    
 4   blood_group           475120 non-null  str    
 5   height_cm             475072 non-null  float64
 6   weight_kg             475048 non-null  float64
 7   bmi                   500000 non-null  float64
 8   patient_city          500000 non-null  str    
 9   patient_state         500000 non-null  str    
 10  pincode               500000 non-null  int64  
 11  marital_status        475000 non-null  str    
 12  primary_diagnosis     500000 non-null  str    
 13  severity              475000 non-null  str    
 14  admission_type        500000 non-null  str    
 15  treatment_t

In [7]:
df_raw.head()

,patient_id,patient_name,age,gender,blood_group,height_cm,weight_kg,bmi,patient_city,patient_state,...,payment_status,smoker,alcohol_consumption,exercise_frequency,diet_type,sleep_hours,satisfaction_rating,complaints,follow_up_required,recommendation_score
0,PAT491537,Kavya Kothari,85,Male,AB+,183.7,145.0,43.0,Kamarhati,Punjab,...,Paid,False,Occasional,3-4 times/week,Vegetarian,20.0,2.0,True,True,1
1,PAT491549,Chaman Thakur,55,Female,AB-,186.7,115.4,33.1,Karaikudi,Nagaland,...,Paid,False,Occasional,3-4 times/week,Vegetarian,5.2,4.0,False,False,9
2,PAT491586,Chanakya Srinivasan,78,Female,O+,141.4,60.6,30.3,Silchar,Goa,...,Partially Paid,False,Occasional,Never,Vegetarian,7.3,4.0,False,False,8
3,PAT491635,Falan Prakash,79,Female,O+,156.2,132.8,54.4,Karaikudi,Madhya Pradesh,...,Pending,False,None,Rarely,Vegetarian,4.8,4.0,False,False,7
4,PAT491656,Urvashi Bhatia,31,Female,A-,127.7,114.6,70.3,Nellore,Jharkhand,...,Paid,False,Occasional,Never,Jain,6.9,5.0,False,False,8


In [8]:
print(f"Total rows : {df_raw.shape[0]}")
print(f"Total columns : {df_raw.shape[1]}")

Total rows : 500000
Total columns : 51


In [9]:
# Are there completely identical rows?
df_raw.duplicated().sum()

np.int64(0)

In [10]:
missing = pd.DataFrame({
    "missing_count" : df_raw.isna().sum(),
    "missing_percentage" : (df_raw.isna().mean() * 100)
})

missing = missing.sort_values("missing_percentage", ascending=False)

print(missing)

                      missing_count  missing_percentage
severity                      25000              5.0000
marital_status                25000              5.0000
alcohol_consumption           25000              5.0000
diet_type                     25000              5.0000
smoker                        25000              5.0000
exercise_frequency            25000              5.0000
insurance_provider            25000              5.0000
weight_kg                     24952              4.9904
height_cm                     24928              4.9856
satisfaction_rating           24882              4.9764
blood_group                   24880              4.9760
sleep_hours                   24877              4.9754
gender                        24876              4.9752
pincode                           0              0.0000
patient_id                        0              0.0000
age                               0              0.0000
patient_name                      0             

In [16]:
col_cat = df_raw.select_dtypes(include = "object").columns
# Select columns with only object datatype aka texts/strings
# .column extracts the name of the columns
for column in col_cat:

    print(df_raw[column].value_counts(dropna=False).head(10)) # Counts missing NaN values as well

C:\Users\sarve\AppData\Local\Temp\ipykernel_31096\2153767293.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  col_cat = df_raw.select_dtypes(include = "object").columns


patient_id
PAT491537    1
PAT491549    1
PAT491586    1
PAT491635    1
PAT491656    1
PAT491664    1
PAT491672    1
PAT491688    1
PAT491713    1
PAT491718    1
Name: count, dtype: int64
patient_name
Sai Dara        13
Akshay Kala     12
Sai Kala        12
Henry Kala      11
Krishna Mani    11
Oscar Kala      11
Prisha Kala     11
Sai Keer        11
Jyoti Mani      11
Anita Kala      11
Name: count, dtype: int64
gender
Male      236689
Female    235935
NaN        24876
MALE         528
F            521
M            505
female       479
male         467
Name: count, dtype: int64
blood_group
O-        59298
A-        59260
O+        59246
AB+       59166
A+        59000
AB-       58913
B+        58885
B-        58852
NaN       24880
B Plus      452
Name: count, dtype: int64
patient_city
Aurangabad        3253
Ghaziabad         3190
Mirzapur          1680
Khammam           1669
Chittoor          1668
Bangalore         1661
Berhampore        1659
Tezpur            1657
Agra              16

In [13]:
df_raw.describe().T # Summary stats for all numerical columns

,count,mean,std,min,25%,50%,75%,max
age,500000.0,50.626646,29.102854,1.00,25.0000,51.000,76.0000,150.000
height_cm,475072.0,160.071630,23.737819,40.00,139.9000,159.900,180.0000,300.000
weight_kg,475048.0,87.803086,37.316521,2.00,56.3000,87.500,118.9000,400.000
bmi,500000.0,36.486084,19.094776,6.30,21.5000,33.500,47.6000,104.200
pincode,500000.0,550342.255060,259520.609031,100000.00,325748.0000,550544.500,774971.0000,999992.000
length_of_stay,500000.0,5.026556,6.708886,-10.00,2.0000,3.000,6.0000,200.000
experience_years,500000.0,18.362250,10.130866,1.00,9.0000,19.000,27.0000,35.000
bed_capacity,500000.0,496.539296,274.849790,52.00,288.0000,483.000,732.0000,998.000
medicine_cost,500000.0,25244.352162,14283.775187,500.06,12888.5275,25260.650,37610.5375,49999.610
lab_cost,500000.0,15151.033564,8573.177024,300.01,7731.9950,15149.275,22577.9200,29999.930


## Business Rule Validation

In [18]:
df_raw.head()

,patient_id,patient_name,age,gender,blood_group,height_cm,weight_kg,bmi,patient_city,patient_state,...,payment_status,smoker,alcohol_consumption,exercise_frequency,diet_type,sleep_hours,satisfaction_rating,complaints,follow_up_required,recommendation_score
0,PAT491537,Kavya Kothari,85,Male,AB+,183.7,145.0,43.0,Kamarhati,Punjab,...,Paid,False,Occasional,3-4 times/week,Vegetarian,20.0,2.0,True,True,1
1,PAT491549,Chaman Thakur,55,Female,AB-,186.7,115.4,33.1,Karaikudi,Nagaland,...,Paid,False,Occasional,3-4 times/week,Vegetarian,5.2,4.0,False,False,9
2,PAT491586,Chanakya Srinivasan,78,Female,O+,141.4,60.6,30.3,Silchar,Goa,...,Partially Paid,False,Occasional,Never,Vegetarian,7.3,4.0,False,False,8
3,PAT491635,Falan Prakash,79,Female,O+,156.2,132.8,54.4,Karaikudi,Madhya Pradesh,...,Pending,False,None,Rarely,Vegetarian,4.8,4.0,False,False,7
4,PAT491656,Urvashi Bhatia,31,Female,A-,127.7,114.6,70.3,Nellore,Jharkhand,...,Paid,False,Occasional,Never,Jain,6.9,5.0,False,False,8


In [23]:
df_raw.loc[
    (df_raw["age"] < 0) | (df_raw["age"] > 120),["patient_name","age"]
].head(5)
# These values are something to ponder upon

,patient_name,age
1695,Damyanti Hegde,136
1862,Aashi Chawla,131
2696,Gayathri Basak,139
2853,Yashawini Prakash,124
3079,Dev Thakkar,122


In [24]:
df_raw.loc[
    (df_raw["height_cm"] < 50) | (df_raw["height_cm"] > 250),["patient_name","height_cm"]
].head(5)

,patient_name,height_cm
188,Harrison Misra,40.0
1278,Wakeeta Chanda,280.0
1393,Sathvik Muni,40.0
1408,Rajeshri Nagi,280.0
1704,Udyati Mandal,280.0


In [25]:
df_raw.loc[
    (df_raw["weight_kg"] < 3) | (df_raw["weight_kg"] > 300),["patient_name","weight_kg"]
].head(5)

,patient_name,weight_kg
1507,Yochana Om,2.0
2907,Bishakha Purohit,2.0
4420,Leela Raval,400.0
4736,Zashil Pal,2.0
6574,Advay Bali,400.0


In [26]:
df_raw.loc[
    (df_raw["length_of_stay"] < 0),["patient_name","length_of_stay"]
].head(5)

,patient_name,length_of_stay
500,Krisha Dyal,-5
961,Urmi Dhar,-5
1427,Isha Lad,-5
1882,Onkar Sur,-5
4012,Udarsh Kade,-10


In [28]:
df_raw.loc[
    (df_raw["sleep_hours"] < 0) | (df_raw["sleep_hours"] > 24), ["patient_name", "sleep_hours"]
].head(5)

,patient_name,sleep_hours
817,Charita Bail,-5.0
1428,Azaan Sibal,25.0
2022,Kashvi Rajagopalan,-5.0
3443,Lakshit Pillai,25.0
4190,Wriddhish Gaba,-5.0


In [30]:
df_raw.loc[
    (df_raw["patient_paid"] < 0) | (df_raw["bill_amount"] < 0), ["patient_name", "bill_amount", "patient_paid"]
].head(5)

,patient_name,bill_amount,patient_paid
78,Chanchal Virk,101946.77,-27010.08
260,Isaiah Prakash,85810.23,-37386.42
330,Yasti Kulkarni,-71156.72,22865.66
578,Vritti Sinha,-47740.20,6247.92
676,Watika Karan,-49576.42,21792.76


In [34]:
calculated_bill = (
    df_raw["medicine_cost"] + df_raw["lab_cost"] + df_raw["room_cost"] + df_raw["doctor_fee"] + df_raw["other_charges"]
)

bill_diff = df_raw["bill_amount"] - calculated_bill

bill_diff.describe()

# For most patients the math is 100% correct. The Quartiles (25%,50%,75%) indicate the bill_diff is 0 down to the last cent. Almost NO bill was marked higher (max) than the sum of the parts.

# Now the negative values - Firstly they are because of bill_amount < sum of parts. (Min) It shows there is a severe outlier wherein the bill paid 4,05,000 less than the estimated count of values. (Mean) It means on an average the bill_amount was 1,100 less than the count of values. (Means Insurance_provider)

count    5.000000e+05
mean    -1.107983e+03
std      1.636423e+04
min     -4.054624e+05
25%      0.000000e+00
50%      0.000000e+00
75%      0.000000e+00
max      5.820766e-11
dtype: float64

In [35]:
df_raw.loc[
    bill_diff.abs() > 1,["patient_id","patient_name"]
].head(5)


,patient_id,patient_name
330,PAT120956,Yasti Kulkarni
578,PAT127940,Vritti Sinha
676,PAT130814,Watika Karan
1264,PAT150218,Chaitanya Mangal
2080,PAT176866,Andrew Verma


## Quantifying the numbers


In [37]:
invalid_age_count = df_raw[(df_raw["age"] < 0) | (df_raw["age"] > 120)]
print(f"Invalid ages : {len(invalid_age_count)}")

Invalid ages : 963


In [38]:
invalid_heights = df_raw[(df_raw["height_cm"] < 50) | (df_raw["height_cm"] > 250)]
print(f"Invalid heights : {len(invalid_heights)}")

Invalid heights : 784


In [39]:
invalid_weights = df_raw[(df_raw["weight_kg"] < 3) | (df_raw["weight_kg"] > 300)]
print(f"Invalid weights : {len(invalid_weights)}")

Invalid weights : 516


In [40]:
invalid_sleep_hours = df_raw[(df_raw["sleep_hours"] < 0) | (df_raw["sleep_hours"] > 24)]
print(f"Invalid sleep hours : {len(invalid_sleep_hours)}")

Invalid sleep hours : 1180


In [41]:
invalid_LOS = df_raw[df_raw["length_of_stay"] < 0]
print(f"Invalid length of stay : {len(invalid_LOS)}")

Invalid length of stay : 526


In [42]:
negative_bill = df_raw[df_raw["bill_amount"] < 0]
print(f"Negative bill amount count : {len(negative_bill)}")

Negative bill amount count : 2500


In [43]:
negative_bill_paid = df_raw[df_raw["patient_paid"] < 0]
print(f"Negative bill amount paid count : {len(negative_bill_paid)}")

Negative bill amount paid count : 2500


## Additional Business Rule Validation

In [44]:
# BMI consistency
expected_bmi = (
    df_raw["weight_kg"] / (df_raw["height_cm"] / 100) ** 2
)
bmi_diff = df_raw["bmi"] - expected_bmi

bmi_diff.describe()
# Key information - 451k rows were evaluated, meaning the rest 48k were NaN values. Meanwhile there are outliers present min (-1493) and max (94)

count    451348.000000
mean         -0.365642
std          13.802964
min       -1493.800000
25%          -0.025000
50%           0.000058
75%           0.025159
max          94.375883
dtype: float64

In [46]:
df_raw.loc[
    bmi_diff.abs() > 0.1,
    [
        "patient_id",
        "height_cm",
        "weight_kg",
        "bmi"
    ]
].head(5)

,patient_id,height_cm,weight_kg,bmi
188,PAT497218,40.0,146.7,74.6
270,PAT499888,250.0,91.6,30.8
445,PAT123233,182.9,250.0,9.0
849,PAT136574,250.0,35.5,11.8
976,PAT140796,172.4,250.0,39.0


In [48]:
insurance_issue_validity = df_raw[(df_raw["bill_amount"] > 0) & (df_raw["insurance_coverage"] > df_raw["bill_amount"])]
print(f"Invalid insurances : {len(insurance_issue_validity)}")

Invalid insurances : 2483


In [54]:
patient_paid_more = df_raw[(df_raw["bill_amount"] > 0) & (df_raw["patient_paid"] > df_raw["bill_amount"])]
print(f"The patients who paid more : {len(patient_paid_more)}")

The patients who paid more : 0


In [50]:
print(
    "Negative patient payments:",
    (df_raw["patient_paid"] < 0).sum()
)

Negative patient payments: 2500


In [55]:
print(df_raw["satisfaction_rating"].value_counts(dropna=False).sort_index())

satisfaction_rating
-2.0       519
 0.0       507
 1.0     29223
 2.0     44690
 3.0     77772
 4.0    158624
 5.0    162309
 6.0       478
 7.0       528
 8.0       468
 NaN     24882
Name: count, dtype: int64


In [57]:
print(df_raw["recommendation_score"].value_counts().sort_index())

recommendation_score
-5       618
 0       607
 1     12471
 2     12419
 3     12587
 4     12359
 5     12343
 6     96936
 7     84630
 8     84299
 9     84629
 10    84827
 11      629
 15      646
Name: count, dtype: int64


In [60]:
pd.crosstab(df_raw["severity"],df_raw["icu_required"],dropna=False)

# Creates a cross-tabular (contingency table) to compare 2 categorial variables : 1st arguments (rows = severity) 2nd arrgument (columns = icu_required)

icu_required,False,True
severity,,
Critical,0,23747
Mild,202857,10740
Moderate,157876,8345
Severe,35713,35722
NaN,20913,4087


In [61]:
df_raw.groupby(
    "severity",
    dropna=False
)["length_of_stay"].agg(
    ["count", "min", "mean", "max"]
)

,count,min,mean,max
severity,,,,
Critical,23747,-10,20.167726,200
Mild,213597,-10,2.126991,200
Moderate,166221,-10,4.609303,200
Severe,71435,-10,9.645776,200
NaN,25000,-10,4.993080,200


In [62]:
print(
    "BMI inconsistencies:",
    (bmi_diff.abs() > 0.1).sum()
)

BMI inconsistencies: 1905


In [66]:
bill_issue = df_raw[
    bill_diff.abs() > 1
]

print(
    "Bill inconsistencies:",
    len(bill_issue)
)

Bill inconsistencies: 2500
